# ABS Mesh Block and SA1 feature extraction for the model sites

This notebook extracts one source-specific feature table for every surveyed site in `data/processed/model_dataset.xlsx`. It uses each site's WGS84 coordinate to identify the containing 2021 ABS Mesh Block (MB) and Statistical Area Level 1 (SA1), then joins the official WA Mesh Block category, area, dwelling count, and usual-resident population.

## What this data source is

This workflow combines two official Australian Bureau of Statistics (ABS) components from the 2021 Census geography:

- The **ASGS Edition 3 (2021) Mesh Block spatial service** supplies the MB and SA1 containing each project coordinate. A Mesh Block is the smallest geographic unit in the ASGS; SA1 is the next Census geography assembled from whole Mesh Blocks.
- The **Census of Population and Housing: Mesh Block Counts, 2021** workbook supplies the dominant MB category, official area, dwelling count, and usual-resident population for Western Australia.

### API used

The notebook uses the official **ABS ArcGIS REST API** for the ASGS 2021 Mesh Block layer:

`https://geo.abs.gov.au/arcgis/rest/services/ASGS2021/MB/MapServer/0/query`

No API key is required. For each site, the notebook sends longitude and latitude as a WGS84 point (`EPSG:4326`) and performs a point–polygon intersection query. It requests only `mb_code_2021` and `sa1_code_2021`, without returning polygon geometry. The API identifies the containing geography; it does **not** return dwelling or population counts. Those attributes are joined afterwards from the local official ABS workbook through `mb_code_2021`.

The spatial service answers **which official area contains this point**. The counts workbook answers **what basic Census context is recorded for that containing Mesh Block**. Together they provide a small-area land-use and residential-context description for each survey site.

## What information we can obtain

For each `site_id`, this source can identify its 2021 MB and SA1, the dominant land-use category of the MB, the MB's official area, its total dwellings, and its usual-resident population. Because Mesh Blocks have unequal areas, the notebook also calculates dwellings per square kilometre and usual residents per square kilometre.

It cannot directly provide pedestrian activity, current population, socioeconomic characteristics, or the number of residents and destinations around the site within a chosen walking buffer. Those questions require other datasets or a separate neighbourhood-buffer method.

## Final output fields and their meaning

The output contains identifiers and QA fields as well as candidate model features. They must not all be passed directly into a model.

| Output field | Type of field | Plain meaning |
|---|---|---|
| `site_id` | Project join key | Identifies the survey site and links this table back to `Model_Data`. |
| `abs_match_status` | QA field | States whether the coordinate produced one usable ABS Mesh Block match. |
| `abs_error_message` | QA field | Explains a failed, missing, or ambiguous match. |
| `abs_mb_code_2021` | Geographic join key | Official code of the containing 2021 Mesh Block. Keep for traceability; do not use it as a continuous numeric predictor. |
| `abs_sa1_code_2021` | Geographic join key | Official code of the containing 2021 SA1. Keep for future SA1-level joins; do not use it as a continuous numeric predictor. |
| `abs_mb_category_2021` | Candidate categorical feature | Dominant MB land-use context, for example Residential, Commercial, Transport, Parkland, or Industrial. It is not a land-use-mix score. |
| `abs_mb_area_sqkm_2021` | Candidate numeric/context field | Official area of the containing MB in square kilometres. It describes polygon size and is also the density denominator. |
| `abs_mb_dwellings_2021` | Candidate numeric feature | Total dwellings recorded in the containing MB. |
| `abs_mb_persons_usual_resident_2021` | Candidate numeric feature | People whose usual residence is in the containing MB; this is not the number of pedestrians observed there. |
| `abs_mb_dwellings_per_sqkm_2021` | Derived candidate numeric feature | Dwelling count divided by MB area, used to compare residential intensity across unequal MB sizes. |
| `abs_mb_persons_per_sqkm_2021` | Derived candidate numeric feature | Usual-resident count divided by MB area, used to compare resident density across unequal MB sizes. |

The six candidate model fields are therefore MB category, MB area, raw dwelling count, raw usual-resident count, dwelling density, and usual-resident density. They are extracted for evaluation, not automatically approved for the final model. With a small modelling sample, correlated raw counts and densities should be compared during feature selection rather than all being forced into the same model.

The data roles are deliberately separated:

- `model_dataset.xlsx` remains the stable foundation table for site identifiers, coordinates, targets, and project QA. It is read but never modified by this notebook.
- `Mesh Block Counts, 2021.xlsx` is the local official ABS count table.
- The ABS ArcGIS REST service supplies the spatial lookup from each point to its 2021 MB and SA1 codes.
- `06_outputs/02_feature_extraction/02_01_haoqi_abs_mesh_block_sa1_features.xlsx` is a derived, source-specific feature table with one row per `site_id`. Its base name matches this notebook, and it can later be joined to the foundation table in modelling code.

All sites are queried, regardless of the target or QA fields, because feature extraction depends only on location. Any training filter belongs in the modelling stage. The output does not copy targets, road attributes, or other foundation-table columns.

Official references used to interpret the data:

- [2021 Census Mesh Block counts](https://www.abs.gov.au/census/guide-census-data/mesh-block-counts/2021)
- [ASGS Edition 3 Mesh Blocks](https://www.abs.gov.au/statistics/standards/australian-statistical-geography-standard-asgs/edition-3-july-2021-june-2026/main-structure-and-greater-capital-city-statistical-areas/mesh-blocks)
- [ASGS Edition 3 Statistical Area Level 1](https://www.abs.gov.au/statistics/standards/australian-statistical-geography-standard-asgs/edition-3-july-2021-june-2026/main-structure-and-greater-capital-city-statistical-areas/statistical-area-level-1)
- [ASGS Edition 3 data services and APIs](https://www.abs.gov.au/statistics/standards/australian-statistical-geography-standard-asgs/edition-3-july-2021-june-2026/access-and-downloads/data-services-and-apis)

Important scientific limitation: these variables describe the **Mesh Block containing the survey coordinate**. They are not counts or densities within a walking-distance buffer around the site. A road point can lie in a Transport, Parkland, Commercial, or other zero-population MB even when residents live nearby.
## Promoted implementation

The reusable implementation now lives in `src/pedflow/context/sources/abs_meshblock_sa1.py`. This notebook remains the scientific explanation, live source review, standalone feature export, and regression reference. It deliberately reads the project foundation workbook itself, whereas the promoted module accepts an already validated `sites` DataFrame and never opens `model_dataset.xlsx`.


In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display


def find_project_root() -> Path:
    """Find the repository root on the host or inside the development container."""
    current = Path.cwd().resolve()
    for path in [current, *current.parents]:
        if (path / "pyproject.toml").is_file() and (path / "src").is_dir():
            return path
    raise FileNotFoundError(
        f"Project root not found above {current}. Expected pyproject.toml and src/."
    )


PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from pedflow.context.contracts import SourceConfig
from pedflow.context.sources import abs_meshblock_sa1 as abs_source
from pedflow.context.validation import (
    validate_and_normalise_sites,
    validate_source_features,
)


MODEL_DATASET_PATH = PROJECT_ROOT / "data" / "processed" / "model_dataset.xlsx"
ABS_COUNTS_PATH = (
    PROJECT_ROOT / "data" / "external_raw" / "Mesh Block Counts, 2021.xlsx"
)
OUTPUT_DIR = PROJECT_ROOT / "06_outputs" / "02_feature_extraction"
NOTEBOOK_STEM = "02_01_haoqi_abs_mesh_block_sa1"
FEATURE_PATH = OUTPUT_DIR / f"{NOTEBOOK_STEM}_features.xlsx"

for required_file in [MODEL_DATASET_PATH, ABS_COUNTS_PATH]:
    if not required_file.is_file():
        raise FileNotFoundError(f"Required input file not found: {required_file}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SOURCE_CONFIG = SourceConfig(
    name=abs_source.SOURCE_NAME,
    enabled=False,
    options={"counts_path": str(ABS_COUNTS_PATH.relative_to(PROJECT_ROOT))},
)

print("Project root :", PROJECT_ROOT)
print("Model input  :", MODEL_DATASET_PATH.relative_to(PROJECT_ROOT))
print("ABS counts   :", ABS_COUNTS_PATH.relative_to(PROJECT_ROOT))
print("Feature file :", FEATURE_PATH.relative_to(PROJECT_ROOT))
print("Source module:", abs_source.__file__)


Project root : /workspace
Model input  : data/processed/model_dataset.xlsx
ABS counts   : data/external_raw/Mesh Block Counts, 2021.xlsx
Feature file : 06_outputs/02_feature_extraction/02_01_haoqi_abs_mesh_block_sa1_features.xlsx
Source module: /workspace/src/pedflow/context/sources/abs_meshblock_sa1.py


## 1. Read the model-site input

The `Model_Data` sheet is the authoritative site list. Only four columns are needed here:

| Input column | Definition and use in this notebook |
|---|---|
| `site_id` | Stable survey-site primary key. It is preserved as text and becomes the only join key for the feature output. |
| `latitude` | WGS84 latitude in decimal degrees; sent to the ABS spatial service. |
| `longitude` | WGS84 longitude in decimal degrees; sent to the ABS spatial service. |
| `coordinate_crs` | Coordinate reference system recorded by the project. The spatial request below requires `EPSG:4326`. |

The checks below are limited to conditions needed for a defensible spatial join: required columns must exist, `site_id` must be complete and unique, coordinates must be numeric and geographically valid, and the CRS must be WGS84. Coordinates are used during processing but are not duplicated in the final feature table.
The notebook now calls the pipeline's shared site validator so this scientific run uses the same input contract as the integrated workflow. The source module still receives only the resulting `sites` DataFrame.


In [2]:
model_data = pd.read_excel(
    MODEL_DATASET_PATH,
    sheet_name="Model_Data",
    dtype={"site_id": "string", "coordinate_crs": "string"},
)

# The shared validator enforces the same site contract used by the integrated pipeline.
sites = validate_and_normalise_sites(model_data)

print(f"Model sites ready for feature extraction: {len(sites):,}")
display(sites.head())


Model sites ready for feature extraction: 49


,site_id,latitude,longitude,coordinate_crs
0,250303-01,-32.866282,115.705163,EPSG:4326
1,250303-02,-33.359642,115.678041,EPSG:4326
2,250303-03,-33.360624,116.154738,EPSG:4326
3,250303-04,-34.238746,116.149491,EPSG:4326
4,250303-05,-32.628918,115.874612,EPSG:4326


## 2. Read the official WA Mesh Block attributes

The workbook `Contents` sheet identifies **Table 5** as Western Australia. Its `Explanatory Notes` sheet defines the source fields. The selected fields and their roles are:

| ABS source field | Official meaning | Role in the feature table |
|---|---|---|
| `MB_CODE_2021` | Eleven-digit 2021 Mesh Block identifier. The first digit is the State or Territory code. | Geographic join and traceability field; do not treat it as a numeric predictor. |
| `MB_CATEGORY_NAME_2021` | Dominant land-use category assigned during MB design, such as Residential, Commercial, Industrial, Parkland, Education, Hospital/Medical, Transport, Primary Production, Water, or Other. | Candidate categorical context feature. It is not a land-use-mix measure. |
| `AREA_ALBERS_SQKM` | Mesh Block area in square kilometres, calculated using the Albers Equal Area Conic projection. | Geographic scale and denominator for the density calculations. |
| `Dwelling` | Total dwellings. ABS counts structures intended and suitable for people to live in on Census Night, including occupied dwellings and most unoccupied private dwellings. | Raw dwelling count and numerator for dwelling density. |
| `Person` | Persons usually resident, counted at the place where they usually live rather than necessarily where they were on Census Night. | Raw usual-resident count and numerator for resident density. It is not observed pedestrian activity. |
| `State` | One-digit State or Territory code; `5` represents Western Australia. | Used to verify the WA rows, then excluded from the feature output. |

The raw counts, area, and category are retained so the density values remain auditable. Density improves comparability across unequal MB areas, but it may be extreme for very small blocks and must be interpreted together with area and category.
The promoted module separates local workbook acquisition (`acquire_or_load_source`) from deterministic cleaning (`prepare_source`). This makes the official field checks testable without an API call.


In [3]:
# External file I/O and deterministic source preparation are separate module steps.
counts_snapshot = abs_source.acquire_or_load_source(SOURCE_CONFIG, PROJECT_ROOT)
wa_counts = abs_source.prepare_source(counts_snapshot, SOURCE_CONFIG)

print(f"WA Mesh Blocks available for joining: {len(wa_counts):,}")
display(wa_counts.head())


WA Mesh Blocks available for joining: 43,322


,abs_mb_code_2021,abs_mb_category_2021,abs_mb_area_sqkm_2021,abs_mb_dwellings_2021,abs_mb_persons_usual_resident_2021
0,50000010000,Residential,0.0357,32,67
1,50000020000,Residential,0.0425,34,81
2,50000030000,Education,0.0816,0,0
3,50000040000,Parkland,0.0581,0,0
4,50000060000,Residential,0.0154,31,44


## 3. Use the official ABS API to identify the containing MB and SA1

The official 2021 Mesh Block ArcGIS layer is queried once for each site coordinate. The service does not require an API key. Only two attributes are requested:

| API field | Meaning and use |
|---|---|
| `mb_code_2021` | Code for the 2021 Mesh Block intersecting the point. It connects the spatial result to the local count table. |
| `sa1_code_2021` | Code for the 2021 SA1 containing that MB. It is retained as a join key for possible future SA1-level data and is not itself a predictor. |

A separate SA1 request is unnecessary because the MB layer already contains the SA1 code. The function preserves `no_match`, `multiple_matches`, and `request_failed` outcomes instead of silently assigning an uncertain geography. This research-scale run intentionally avoids a production caching or retry system; failed requests remain visible and can be rerun.
The notebook calls the promoted `query_meshblock_codes` acquisition function. Credentials are not required or stored, and request failures expose a safe status rather than raw response content.


In [4]:
print(f"Querying the ABS spatial service for {len(sites):,} sites...")
api_matches = abs_source.query_meshblock_codes(sites, SOURCE_CONFIG)

display(api_matches.head())
display(api_matches["abs_match_status"].value_counts(dropna=False).to_frame("sites"))


Querying the ABS spatial service for 49 sites...


,site_id,abs_match_status,abs_error_message,abs_mb_code_2021,abs_sa1_code_2021
0,250303-01,matched,<NA>,50391642000,50102101608
1,250303-02,matched,<NA>,50043794000,50102101101
2,250303-03,matched,<NA>,50366540000,50102100909
3,250303-04,matched,<NA>,50196282000,50103101903
4,250303-05,matched,<NA>,50227251000,50201102910


,sites
abs_match_status,
matched,49


## 4. Build and save the source-specific feature table

The API result is joined to the local WA count table through `abs_mb_code_2021`. The final workbook contains one sheet, `ABS_MB_SA1_Features`, and one row per input `site_id`. Its columns are:

| Output column | Interpretation |
|---|---|
| `site_id` | Project primary key used to join this feature table back to `Model_Data`. |
| `abs_match_status`, `abs_error_message` | Spatial/API result and any reason a usable feature row could not be produced. These are QA fields, not predictors. |
| `abs_mb_code_2021`, `abs_sa1_code_2021` | ABS geographic identifiers for traceability and later joins; not continuous predictors. |
| `abs_mb_category_2021` | Dominant category of the containing MB. |
| `abs_mb_area_sqkm_2021` | Official area of the containing MB, not the area of a site-centred buffer. |
| `abs_mb_dwellings_2021`, `abs_mb_persons_usual_resident_2021` | Official raw dwelling and usual-resident counts in the containing MB. |
| `abs_mb_dwellings_per_sqkm_2021`, `abs_mb_persons_per_sqkm_2021` | Raw count divided by official MB area where area is greater than zero. |

The output is written to `06_outputs/02_feature_extraction/02_01_haoqi_abs_mesh_block_sa1_features.xlsx`. The shared base name keeps the extraction notebook and its derived feature table visibly paired, while the `06_outputs` directory keeps reviewed results separate from source notebooks. Dataset definitions remain in this notebook rather than being repeated as data columns or as another metadata output. The final code confirms that the row count and unique `site_id` values still match the foundation table.
The final cell calls the deterministic `build_features` function and constructs a `FeatureResult` containing the source version, safe provenance metadata, and warnings. Before saving, it compares identifiers, categories, statuses, errors, and counts exactly with an existing reviewed workbook; area and density values use `rtol=1e-10` and `atol=1e-10` only for floating-point representation differences.


In [5]:
# Deterministic calculation is separate from external source acquisition.
features = abs_source.build_features(sites, api_matches, wa_counts)
feature_result = abs_source.make_feature_result(
    sites,
    features,
    counts_snapshot,
    wa_counts,
    SOURCE_CONFIG,
)
features = validate_source_features(
    source_name=feature_result.source_name,
    features=feature_result.features,
    expected_site_ids=sites["site_id"],
    column_prefixes=abs_source.DEFINITION.column_prefixes,
)

# If a reviewed workbook already exists, prove that promotion did not change it
# before overwriting that standalone notebook output.
if FEATURE_PATH.is_file():
    reviewed = pd.read_excel(
        FEATURE_PATH,
        sheet_name="ABS_MB_SA1_Features",
        dtype={
            "site_id": "string",
            "abs_mb_code_2021": "string",
            "abs_sa1_code_2021": "string",
        },
    )
    actual_exact = features.loc[:, list(abs_source.EXACT_REGRESSION_COLUMNS)]
    reviewed_exact = reviewed.loc[:, list(abs_source.EXACT_REGRESSION_COLUMNS)]
    actual_exact = actual_exact.astype("object").where(
        actual_exact.notna(), "<MISSING>"
    )
    reviewed_exact = reviewed_exact.astype("object").where(
        reviewed_exact.notna(), "<MISSING>"
    )
    pd.testing.assert_frame_equal(
        actual_exact.reset_index(drop=True),
        reviewed_exact.reset_index(drop=True),
        check_dtype=False,
        check_exact=True,
    )
    np.testing.assert_allclose(
        features.loc[:, list(abs_source.FLOAT_REGRESSION_COLUMNS)].to_numpy(
            dtype=float
        ),
        reviewed.loc[:, list(abs_source.FLOAT_REGRESSION_COLUMNS)].to_numpy(
            dtype=float
        ),
        rtol=abs_source.REGRESSION_RTOL,
        atol=abs_source.REGRESSION_ATOL,
        equal_nan=True,
    )
    print("Regression check: promoted output matches the reviewed workbook.")

features.to_excel(
    FEATURE_PATH,
    sheet_name="ABS_MB_SA1_Features",
    index=False,
)

display(features)
display(features["abs_match_status"].value_counts(dropna=False).to_frame("sites"))
print("Source version:", feature_result.source_version)
for warning in feature_result.warnings:
    print("Warning:", warning)
print("Saved:", FEATURE_PATH.relative_to(PROJECT_ROOT))


Regression check: promoted output matches the reviewed workbook.


,site_id,abs_match_status,abs_error_message,abs_mb_code_2021,abs_sa1_code_2021,abs_mb_category_2021,abs_mb_area_sqkm_2021,abs_mb_dwellings_2021,abs_mb_persons_usual_resident_2021,abs_mb_dwellings_per_sqkm_2021,abs_mb_persons_per_sqkm_2021
0,250303-01,matched,<NA>,50391642000,50102101608,Residential,7.9418,35,74,4.407061,9.317787
1,250303-02,matched,<NA>,50043794000,50102101101,Industrial,0.3167,0,0,0.0,0.0
2,250303-03,matched,<NA>,50366540000,50102100909,Transport,0.1362,0,0,0.0,0.0
3,250303-04,matched,<NA>,50196282000,50103101903,Commercial,0.0568,0,0,0.0,0.0
4,250303-05,matched,<NA>,50227251000,50201102910,Commercial,0.0616,0,0,0.0,0.0
5,250303-06,matched,<NA>,50143360000,50102101411,Residential,0.0455,36,83,791.208791,1824.175824
6,250303-07,matched,<NA>,50099181000,50103101805,Commercial,0.0779,0,0,0.0,0.0
7,250303-08,matched,<NA>,50062942000,50102100704,Residential,0.1137,12,27,105.540897,237.467018
8,250303-09,matched,<NA>,50333692000,50102101602,Parkland,0.0938,0,0,0.0,0.0
9,250303-10,matched,<NA>,50033911000,50103101712,Commercial,0.0325,0,0,0.0,0.0


,sites
abs_match_status,
matched,49


Source version: ABS Mesh Block Counts 2021 + ASGS Edition 3 (2021) MB layer 0
Saved: 06_outputs/02_feature_extraction/02_01_haoqi_abs_mesh_block_sa1_features.xlsx
